# 03 — Tokens and Embeddings

**Description:** Turn text into token IDs, build an embedding matrix, look up vectors with NumPy and PyTorch, and track shapes through batches and sequences.
**Level:** Beginner
**Tags:** Language Models, Tokenization, Embeddings, NumPy, PyTorch

A neural network cannot multiply words by weight matrices. It needs numbers. Language-model input therefore passes through two important conversions:

$$\text{text} \rightarrow \text{tokens} \rightarrow \text{token IDs} \rightarrow \text{embedding vectors}$$

In this notebook, every conversion stays visible. By the end, you will be able to:

- tokenize a small text corpus;
- build a vocabulary and map tokens to integer IDs;
- use IDs to select rows from an embedding matrix;
- predict embedding-output shapes for sequences and batches; and
- use `torch.nn.Embedding` and inspect its trainable parameters.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=3, suppress=True)
torch.set_printoptions(precision=3, sci_mode=False)
rng = np.random.default_rng(7)
torch.manual_seed(7)

## 1. From text to tokens

A **tokenizer** splits text into a sequence of discrete pieces called tokens. Tokens are not necessarily words: modern language models often use common words as single tokens and split rarer words into subword pieces.

To keep this notebook transparent, our tokenizer lowercases text and extracts words and punctuation. It is useful for learning, but it is not meant to handle every language or writing system.

In [ ]:
def tokenize(text):
    """Split lowercase text into word and punctuation tokens."""
    return re.findall(r"[a-z]+|[^\w\s]", text.lower())

text = "The robot learns from examples."
tokens = tokenize(text)

print("text:  ", text)
print("tokens:", tokens)
print("count: ", len(tokens))

### Your turn: test the tokenizer

Change the sentence and predict the output before running the cell. Try repeated punctuation, capitalization, a number, or a hyphenated word. Which cases reveal limitations in our pattern?

In [ ]:
your_text = "Tokens, tokens everywhere!"  # Edit me
tokenize(your_text)

## 2. A vocabulary assigns an ID to each token

A **vocabulary** is the finite set of tokens a model knows. Each token receives a unique integer ID. The ID is only an address—it does not mean that token 8 is semantically greater than token 3.

We will reserve three special tokens:

- `<PAD>` fills unused positions when sequences have different lengths;
- `<UNK>` represents a token outside the vocabulary;
- `<EOS>` marks the end of a sequence.

In [ ]:
corpus = [
    "the cat chases the mouse",
    "the dog chases the ball",
    "the kitten sees the cat",
    "the puppy sees the dog",
    "the cat likes warm milk",
    "the dog likes long walks",
]

special_tokens = ["<PAD>", "<UNK>", "<EOS>"]
corpus_tokens = [token for sentence in corpus for token in tokenize(sentence)]
vocabulary = special_tokens + sorted(set(corpus_tokens))
token_to_id = {token: index for index, token in enumerate(vocabulary)}
id_to_token = {index: token for token, index in token_to_id.items()}

print("vocabulary size:", len(vocabulary))
print(token_to_id)

## 3. Encoding and decoding

**Encoding** maps tokens to IDs. **Decoding** maps IDs back to readable tokens. An unknown input token maps to `<UNK>` instead of causing an error.

Adding `<EOS>` makes the end explicit. This matters during generation: a model can predict `<EOS>` to say that it is finished.

In [ ]:
def encode(text, add_eos=True):
    ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in tokenize(text)]
    if add_eos:
        ids.append(token_to_id["<EOS>"])
    return ids

def decode(ids):
    return [id_to_token[int(token_id)] for token_id in ids]

sentence = "the cat sees a dragon"
token_ids = encode(sentence)
print("text:   ", sentence)
print("IDs:    ", token_ids)
print("decoded:", decode(token_ids))

### Inspect the representation

At this stage, a sequence of $L$ tokens has become an integer vector with shape `(L,)`. The length depends on tokenization and whether `<EOS>` is included. Encode several inputs and compare their shapes.

In [ ]:
for sentence in ["the cat", "the dog likes walks", "a dragon dances"]:
    ids = np.array(encode(sentence))
    print(f"{sentence!r:24s} → {ids}  shape={ids.shape}")

## 4. Why IDs are not features

It would be a mistake to feed token IDs into a layer as ordinary magnitudes. The IDs are arbitrary labels: changing the vocabulary order changes the numbers without changing language.

A **one-hot vector** is a safer conceptual representation. It has one position per vocabulary item, with a 1 at the token's ID and 0 everywhere else. Two different tokens are equally distinct, regardless of their IDs.

In [ ]:
def one_hot(token_id, vocab_size):
    vector = np.zeros(vocab_size)
    vector[token_id] = 1.0
    return vector

cat_id = token_to_id["cat"]
cat_one_hot = one_hot(cat_id, len(vocabulary))

print("cat ID:       ", cat_id)
print("one-hot shape:", cat_one_hot.shape)
print("nonzero index:", np.flatnonzero(cat_one_hot))
print("sum:          ", cat_one_hot.sum())

## 5. The embedding matrix

One-hot vectors are large and mostly zeros. An **embedding matrix** stores one short, dense vector for every vocabulary item. Its shape is:

$$({\text{vocabulary size}}, {\text{embedding dimension}})$$

Each row belongs to one token. Looking up token ID `i` means selecting row `i`. We will start with four-dimensional random vectors so every value remains easy to inspect.

In [ ]:
vocab_size = len(vocabulary)
embedding_dim = 4
embedding_matrix = rng.normal(0, 0.5, size=(vocab_size, embedding_dim))

print("matrix shape:", embedding_matrix.shape)
print("cat row:     ", embedding_matrix[cat_id])
print("dog row:     ", embedding_matrix[token_to_id["dog"]])

## 6. A lookup is row selection

NumPy's integer indexing can look up an entire token sequence at once. If the ID input has shape `(sequence_length,)`, the output has shape `(sequence_length, embedding_dim)`.

The lookup does not average the sequence. It preserves token order: output row 0 corresponds to input token 0, output row 1 to input token 1, and so on.

In [ ]:
sentence = "the cat chases the mouse"
ids = np.array(encode(sentence))
embedded = embedding_matrix[ids]

print("tokens:         ", decode(ids))
print("ID shape:       ", ids.shape)
print("embedding shape:", embedded.shape)
print("first token ID: ", ids[0])
print("first vector:   ", embedded[0])

### Verify one-hot multiplication

Selecting row `i` is mathematically equivalent to multiplying token `i`'s one-hot vector by the embedding matrix. Libraries use direct lookup because it avoids constructing the large one-hot vector.

In [ ]:
dog_id = token_to_id["dog"]
by_lookup = embedding_matrix[dog_id]
by_multiplication = one_hot(dog_id, vocab_size) @ embedding_matrix

print("lookup:        ", by_lookup)
print("multiplication:", by_multiplication)
print("match:         ", np.allclose(by_lookup, by_multiplication))

## 7. Shape rule for embedding lookup

An embedding layer appends one new axis of size `embedding_dim` to the input-ID shape:

| ID input | ID shape | Embedding output shape |
|---|---:|---:|
| one token | `()` | `(embedding_dim,)` |
| one sequence | `(sequence,)` | `(sequence, embedding_dim)` |
| batch of sequences | `(batch, sequence)` | `(batch, sequence, embedding_dim)` |

A useful shorthand is: `(*input_shape) → (*input_shape, embedding_dim)`.

In [ ]:
one_id = np.array(token_to_id["cat"])
sequence_ids = np.array([token_to_id["cat"], token_to_id["likes"]])
batch_ids = np.array([
    [token_to_id["cat"], token_to_id["likes"]],
    [token_to_id["dog"], token_to_id["likes"]],
])

for name, value in [("one token", one_id), ("sequence", sequence_ids), ("batch", batch_ids)]:
    print(f"{name:10s}: IDs {value.shape!s:8s} → embeddings {embedding_matrix[value].shape}")

### Your turn: predict before running

Suppose the ID tensor has shape `(3, 5)` and the embedding dimension is 8. What will the output shape be? Change the values below to test other shapes.

In [ ]:
practice_ids = np.zeros((3, 5), dtype=int)
practice_matrix = np.zeros((vocab_size, 8))
practice_output = practice_matrix[practice_ids]

print("input shape: ", practice_ids.shape)
print("output shape:", practice_output.shape)

## 8. The same lookup with PyTorch

`nn.Embedding(num_embeddings, embedding_dim)` stores a trainable embedding matrix. It expects integer token IDs with dtype `torch.long`. Calling the layer performs the lookup and preserves the input axes.

In [ ]:
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
ids_torch = torch.tensor(encode("the cat likes milk"), dtype=torch.long)
vectors_torch = embedding(ids_torch)

print(embedding)
print("ID dtype:       ", ids_torch.dtype)
print("ID shape:       ", ids_torch.shape)
print("embedding shape:", vectors_torch.shape)
print("matrix shape:   ", embedding.weight.shape)

### Inspect the correspondence

The first output vector must equal the embedding-matrix row selected by the first ID. Verify this directly. Notice that repeated IDs retrieve identical vectors because a token has one shared embedding row.

In [ ]:
first_id = ids_torch[0]
print("first token:   ", id_to_token[first_id.item()])
print("layer output:  ", vectors_torch[0])
print("matrix row:    ", embedding.weight[first_id])
print("same values:   ", torch.allclose(vectors_torch[0], embedding.weight[first_id]))

## 9. Batches need equal sequence lengths

Two sentences often contain different numbers of tokens. To place them in one rectangular tensor, we pad shorter sequences to the longest length using `<PAD>`. A **padding mask** records which positions contain real tokens.

Padding is a storage convenience, not meaningful text. Later model operations should ignore padded positions when appropriate.

In [ ]:
def pad_batch(sentences):
    encoded = [encode(sentence) for sentence in sentences]
    max_length = max(map(len, encoded))
    pad_id = token_to_id["<PAD>"]
    padded = [ids + [pad_id] * (max_length - len(ids)) for ids in encoded]
    ids = torch.tensor(padded, dtype=torch.long)
    mask = ids != pad_id
    return ids, mask

sentences = ["the cat", "the dog likes long walks", "the mouse sees the cat"]
batch_ids, padding_mask = pad_batch(sentences)

print("IDs shape: ", batch_ids.shape)
print(batch_ids)
print("mask shape:", padding_mask.shape)
print(padding_mask)

### Embed the whole batch

The batch begins with shape `(batch, sequence)`. Embedding appends the feature dimension, producing `(batch, sequence, embedding_dim)`. Index `[1, 2]` selects the vector for sequence 1, token position 2.

In [ ]:
batch_vectors = embedding(batch_ids)

print("IDs:       ", batch_ids.shape)
print("embeddings:", batch_vectors.shape)
print("selected ID:", batch_ids[1, 2].item())
print("token:      ", id_to_token[batch_ids[1, 2].item()])
print("vector:     ", batch_vectors[1, 2])

## 10. Embeddings are learned parameters

The random vectors currently have no useful meaning. During language-model training, gradients update the rows used in the batch. Tokens that help solve similar prediction problems can develop similar vectors.

`padding_idx` is a useful PyTorch option: it keeps the padding row fixed at zero so padding contributes no learned content.

In [ ]:
padding_id = token_to_id["<PAD>"]
trainable_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_id)

print("parameter matrix:", trainable_embedding.weight.shape)
print("parameter count: ", trainable_embedding.weight.numel())
print("requires grad:   ", trainable_embedding.weight.requires_grad)
print("padding vector:  ", trainable_embedding.weight[padding_id])

## 11. A tiny learning demonstration

We can deliberately arrange six token vectors so related animals begin near each other. Real embeddings are not hand-designed this way—they acquire structure through training—but this tiny example previews the geometry explored in Notebook 04.

The two dimensions do not have official names. Their meaning comes from how tokens are positioned relative to one another.

In [ ]:
demo_tokens = ["cat", "kitten", "dog", "puppy", "milk", "walks"]
demo_vectors = np.array([
    [ 1.0,  0.8],  # cat
    [ 1.2,  1.0],  # kitten
    [-1.0,  0.8],  # dog
    [-1.2,  1.0],  # puppy
    [ 0.8, -1.0],  # milk
    [-0.8, -1.0],  # walks
])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(demo_vectors[:, 0], demo_vectors[:, 1], s=80, color="#4C78A8")
for token, (x, y) in zip(demo_tokens, demo_vectors):
    ax.annotate(token, (x, y), xytext=(6, 5), textcoords="offset points")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set(title="A tiny, illustrative embedding space", xlabel="dimension 1", ylabel="dimension 2")
plt.show()

## 12. Common mistakes

- **Treating IDs as magnitudes:** token IDs are row addresses, not meaningful numerical features.
- **Using floating-point IDs:** PyTorch embedding indices must be integer tensors, normally `torch.long`.
- **Swapping dimensions:** the matrix shape is `(vocab_size, embedding_dim)`.
- **Losing the sequence axis:** lookup returns one vector per input position; it does not combine the sequence.
- **Ignoring padding:** padded positions need a consistent ID and often a mask.
- **Assuming random embeddings have semantics:** useful geometry must be learned from an objective and data.

## 13. Challenges

1. **Tokenizer:** Modify `tokenize` so numbers remain tokens instead of being discarded. Test `"version 3 has 12 layers"`.
2. **Vocabulary:** Build a vocabulary ordered by token frequency instead of alphabetically. Keep special tokens first.
3. **Shapes:** Create a batch of 4 sequences, each 7 tokens long, with embedding dimension 16. Predict and verify every shape.
4. **Parameter count:** How many parameters does an embedding layer with 50,000 tokens and 768 dimensions contain? Approximately how much memory would those 32-bit floats require?
5. **Lookup equivalence:** One-hot encode an entire sequence as a matrix and multiply it by `embedding_matrix`. Confirm that it matches direct indexing.
6. **Padding:** Use the mask to compute the mean embedding of each sentence while excluding padded positions.

In [ ]:
# Challenge workspace: compute a padding-aware mean for each sequence.
vectors = trainable_embedding(batch_ids)
mask_as_numbers = padding_mask.unsqueeze(-1).float()
masked_sum = (vectors * mask_as_numbers).sum(dim=1)
real_token_counts = mask_as_numbers.sum(dim=1)
mean_vectors = masked_sum / real_token_counts

print("token vectors:", vectors.shape)
print("mean vectors: ", mean_vectors.shape)
mean_vectors

## Takeaways

- Tokenization converts text into discrete pieces.
- A vocabulary maps each known token to an arbitrary integer ID.
- An embedding matrix contains one dense vector per vocabulary item.
- Embedding lookup is row selection, equivalent to one-hot matrix multiplication.
- Lookup preserves every input axis and appends the embedding dimension.
- `nn.Embedding` stores trainable vectors that acquire useful structure during training.
- Padding and masks let unequal-length sequences fit into rectangular batches.

**Next:** *04 — Understanding Embedding Space* will measure vector similarity and explore semantic directions.